In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../../data/Clean_BNPParibas_Data.csv")

X = df.drop(columns=["contract_type", "customer_id"])
y = df["contract_type"]

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

Numeric features:
['age', 'tenure_months', 'monthly_charges', 'total_charges', 'support_tickets', 'churn']

Categorical features:
['internet_service', 'payment_method']


In [5]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

svm_classifier = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", SVC(kernel="rbf"))
    ]
)

svm_classifier.fit(
    X_train,
    y_train
)

svm_predictions = svm_classifier.predict(X_test)

In [6]:
from sklearn.neighbors import KNeighborsClassifier

knn_classifier = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", KNeighborsClassifier(n_neighbors=5))
    ]
)

knn_classifier.fit(
    X_train,
    y_train
)

knn_predictions = knn_classifier.predict(X_test)

In [7]:
from sklearn.ensemble import GradientBoostingClassifier

gradient_boosting_classifier = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            GradientBoostingClassifier(
                random_state=42
            )
        )
    ]
)

gradient_boosting_classifier.fit(
    X_train,
    y_train
)

gb_predictions = (
    gradient_boosting_classifier.predict(X_test)
)

In [8]:
from sklearn.metrics import accuracy_score

classification_results = pd.DataFrame({
    "Model": [
        "SVM",
        "KNN",
        "Gradient Boosting"
    ],
    "Accuracy": [
        accuracy_score(y_test, svm_predictions),
        accuracy_score(y_test, knn_predictions),
        accuracy_score(y_test, gb_predictions)
    ]
})

display(
    classification_results.sort_values(
        "Accuracy",
        ascending=False
    )
)

,Model,Accuracy
2,Gradient Boosting,0.590164
0,SVM,0.562842
1,KNN,0.557377
